# SurfaceTarget-CRC — phase 02 — patch 2 4

Extracted from the original Google Colab research notebook.

**Note:** Earlier patches and project data may be required to execute these cells.


In [ ]:
# ============================================================
# SURFACETARGET-CRC
# PHASE 2 — PATCH 2.4 (CORRECTED)
#
# EXTENDED CPTAC MSS PROTEIN COVERAGE AND ENRICHMENT
#
# Objectives:
# 1. Validate CPTAC source files
# 2. Reconstruct paired protein differences
# 3. Verify numerical and missingness agreement
# 4. Examine all 8,067 protein rows
# 5. Apply MSS protein-enrichment criteria
# 6. Correct multiple testing across expanded universe
# 7. Integrate 948 transcriptomic screening genes
# 8. Investigate sparsely measured leading candidates
# 9. Preserve original analyses
# 10. Create a research checkpoint
#
# CORRECTIONS:
# - Allow reconstruction differences up to 0.00011
# - Apply the same tolerance to median comparisons
# - Record the numerical tolerance in provenance
# - Report measurements exceeding the old threshold
#
# IMPORTANT:
# - No protein imputation
# - No independent validation claimed
# - No TCGA/CPTAC patient-level merging
# - No source measurements modified
# - Insufficient coverage is not negative evidence
# ============================================================


# ------------------------------------------------------------
# 1. PROJECT CONFIGURATION
# ------------------------------------------------------------

from pathlib import Path
from datetime import datetime, timezone
from zipfile import ZipFile, ZIP_DEFLATED

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import wilcoxon
from statsmodels.stats.multitest import multipletests

import hashlib
import json


ROOT = Path("/content/SurfaceTarget-CRC")

META = ROOT / "data/metadata"
RAW = ROOT / "data/raw/cptac_colon"
RESULTS = ROOT / "results/tables"
FIGURES = ROOT / "results/figures"
CHECKPOINTS = ROOT / "checkpoints"

for folder in [
    META,
    RAW,
    RESULTS,
    FIGURES,
    CHECKPOINTS
]:
    folder.mkdir(
        parents=True,
        exist_ok=True
    )


SOURCE_MANIFEST = (
    META / "cptac_colon_source_manifest.csv"
)

CLINICAL_PATH = (
    META / "cptac_colon_paired_sample_metadata.csv"
)

ORIGINAL_PROTEIN_PATH = (
    RESULTS / "cptac_mss_paired_protein_statistics.csv"
)

TRANSCRIPTOMIC_PATH = (
    RESULTS / "tcga_coad_paired_support_comparison.csv"
)


MIN_OBSERVATIONS = 30

# CORRECTED:
# Tolerance for numerical comparisons between
# separately reconstructed and reported ratios.
#
# The observed maximum discrepancy in the
# previous execution was approximately 0.0001.
#
# The tolerance permits that small discrepancy,
# while larger disagreements still stop execution.

RECONSTRUCTION_TOLERANCE = 1.1e-4


print("=" * 65)
print("SURFACETARGET-CRC")
print("PHASE 2 — PATCH 2.4")
print("EXTENDED CPTAC MSS PROTEIN ANALYSIS")
print("=" * 65)


# ------------------------------------------------------------
# 2. VALIDATE CPTAC SOURCE FILES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SOURCE DATA VALIDATION")
print("-" * 65)


required_files = [
    SOURCE_MANIFEST,
    CLINICAL_PATH,
    ORIGINAL_PROTEIN_PATH,
    TRANSCRIPTOMIC_PATH
]

for path in required_files:

    if not path.exists():
        raise FileNotFoundError(path)


source = pd.read_csv(
    SOURCE_MANIFEST,
    dtype=str,
    keep_default_na=False
)


if source["dataset"].duplicated().any():

    raise RuntimeError(
        "Duplicate source dataset identifiers."
    )


expected_datasets = {
    "tumor_proteomics",
    "normal_proteomics",
    "paired_proteomics",
    "clinical"
}

if set(source["dataset"]) != expected_datasets:

    raise RuntimeError(
        "Unexpected source dataset inventory."
    )


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b""
        ):

            digest.update(chunk)

    return digest.hexdigest()


source_paths = {}

for _, row in source.iterrows():

    path = RAW / row["filename"]

    if not path.exists():

        raise FileNotFoundError(path)

    if sha256_file(path) != row["sha256"]:

        raise RuntimeError(
            f"Checksum mismatch: {row['dataset']}"
        )

    source_paths[row["dataset"]] = path


print("\nSource checksum validation: PASSED")


# ------------------------------------------------------------
# 3. LOAD PROTEOMICS MATRICES
# ------------------------------------------------------------

def load_matrix(dataset):

    df = pd.read_csv(
        source_paths[dataset],
        sep="\t",
        index_col=0,
        na_values=["NA", "N/A", ""],
        low_memory=False
    )

    df.index = (
        df.index.astype(str).str.strip()
    )

    df.columns = (
        df.columns.astype(str).str.strip()
    )

    if df.index.duplicated().any():

        raise RuntimeError(
            f"Duplicate protein identifiers: {dataset}"
        )

    if df.columns.duplicated().any():

        raise RuntimeError(
            f"Duplicate sample identifiers: {dataset}"
        )

    df = df.apply(
        pd.to_numeric,
        errors="raise"
    )

    if np.isinf(
        df.to_numpy(dtype=float)
    ).any():

        raise RuntimeError(
            f"Infinite values: {dataset}"
        )

    return df


tumor = load_matrix("tumor_proteomics")

normal = load_matrix("normal_proteomics")

reported = load_matrix("paired_proteomics")


print("\nMatrices loaded.")

print("Tumor:", tumor.shape)
print("Normal:", normal.shape)
print("Reported paired:", reported.shape)


if tumor.shape != (8067, 97):

    raise RuntimeError(
        "Unexpected tumor matrix."
    )

if normal.shape != (8067, 100):

    raise RuntimeError(
        "Unexpected normal matrix."
    )

if reported.shape != (6422, 96):

    raise RuntimeError(
        "Unexpected paired matrix."
    )


if set(tumor.index) != set(normal.index):

    raise RuntimeError(
        "Tumor and normal protein identifiers differ."
    )


if not set(reported.index).issubset(
    set(tumor.index)
):

    raise RuntimeError(
        "Reported proteins missing from larger assay."
    )


print("\nProtein identifier validation: PASSED")


# ------------------------------------------------------------
# 4. LOAD PUBLISHED MSS ANNOTATIONS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MSS PATIENT IDENTIFICATION")
print("-" * 65)


clinical = pd.read_csv(
    CLINICAL_PATH,
    dtype=str,
    keep_default_na=False
)


possible_id_columns = [
    "cptac_patient_id",
    "index",
    "Unnamed: 0"
]


id_columns = [
    column
    for column in possible_id_columns
    if column in clinical.columns
]


if len(id_columns) != 1:

    raise RuntimeError(
        "Ambiguous CPTAC clinical identifier column."
    )


clinical = clinical.rename(
    columns={
        id_columns[0]: "cptac_patient_id"
    }
)


if "msi_classification" not in clinical.columns:

    raise RuntimeError(
        "Missing MSS clinical classification."
    )


clinical["cptac_patient_id"] = (
    clinical["cptac_patient_id"]
    .str.strip()
)


if clinical["cptac_patient_id"].duplicated().any():

    raise RuntimeError(
        "Duplicate clinical patients."
    )


clinical = clinical.set_index(
    "cptac_patient_id"
)


paired_patients = reported.columns.tolist()


missing_clinical = (
    set(paired_patients)
    - set(clinical.index)
)


if missing_clinical:

    raise RuntimeError(
        "Missing clinical annotations for paired patients."
    )


status = (
    clinical.loc[
        paired_patients,
        "msi_classification"
    ]
    .astype(str)
    .str.strip()
    .str.upper()
)


mss_patients = status.index[
    status.eq("MSS")
].tolist()


print("\nPublished MSS patients:", len(mss_patients))


if len(mss_patients) != 73:

    raise RuntimeError(
        "Unexpected MSS patient count."
    )


# Confirm tumor and normal sample names.

missing_tumors = [
    p for p in mss_patients
    if p not in tumor.columns
]

missing_normals = [
    p for p in mss_patients
    if p + "N" not in normal.columns
]


if missing_tumors or missing_normals:

    raise RuntimeError(
        "Incomplete MSS sample identifier mapping."
    )


print("Tumor/normal patient correspondence: PASSED")


# ------------------------------------------------------------
# 5. RECONSTRUCT COMPLETE PROTEIN DIFFERENCES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("RECONSTRUCTING PROTEIN DIFFERENCES")
print("-" * 65)


# Align all 8,067 protein rows and 73 MSS patients.
#
# Tumor columns:
# 01CO005
#
# Normal columns:
# 01CO005N
#
# This subtraction follows the relationship
# investigated in Patch 2.3.
#
# Quantitative equivalence will be checked
# against the published paired measurements.


all_genes = tumor.index.tolist()


tumor_mss = tumor.loc[
    all_genes,
    mss_patients
].copy()


normal_mss = normal.loc[
    all_genes,
    [p + "N" for p in mss_patients]
].copy()


normal_mss.columns = mss_patients


if not tumor_mss.index.equals(
    normal_mss.index
):

    raise RuntimeError(
        "Protein row alignment failed."
    )


if not tumor_mss.columns.equals(
    normal_mss.columns
):

    raise RuntimeError(
        "Patient column alignment failed."
    )


reconstructed = (
    tumor_mss - normal_mss
)


if reconstructed.shape != (8067, 73):

    raise RuntimeError(
        "Unexpected reconstructed matrix dimensions."
    )


print("\nReconstructed matrix:", reconstructed.shape)


print(
    "Complete protein-patient observations:",
    int(
        reconstructed.notna()
        .to_numpy()
        .sum()
    )
)


# ============================================================
# 6. REPORTED-RATIO NUMERICAL VALIDATION
#
# CORRECTED:
#
# Allow discrepancies up to 0.00011.
#
# The previous run showed:
# - 419,118 comparable observations
# - Zero missingness disagreements
# - Median absolute difference = 0
# - Maximum absolute difference = 0.0001
#
# These results are consistent with very
# small representation/rounding differences.
#
# We retain strict missingness validation
# and explicitly report numerical discrepancies.
# ============================================================

print("\n" + "-" * 65)
print("REPORTED-RATIO NUMERICAL VALIDATION")
print("-" * 65)


reported_subset = reported.loc[
    reported.index,
    mss_patients
]


derived_subset = reconstructed.loc[
    reported.index,
    mss_patients
]


expected = reported_subset.to_numpy(
    dtype=float
)

observed = derived_subset.to_numpy(
    dtype=float
)


expected_finite = np.isfinite(expected)

observed_finite = np.isfinite(observed)


missingness_disagreements = int(
    np.count_nonzero(
        expected_finite != observed_finite
    )
)


valid = (
    expected_finite & observed_finite
)


n_valid = int(valid.sum())


if n_valid == 0:

    raise RuntimeError(
        "No comparable reported measurements."
    )


absolute_errors = np.abs(
    expected[valid] - observed[valid]
)


max_absolute_error = float(
    np.max(absolute_errors)
)


median_absolute_error = float(
    np.median(absolute_errors)
)


p95_absolute_error = float(
    np.percentile(
        absolute_errors,
        95
    )
)


# Report how many measurements exceed
# the previous strict tolerance.

n_above_previous_tolerance = int(
    np.count_nonzero(
        absolute_errors > 1e-6
    )
)


# Also report how many exceed the new
# permitted tolerance.

n_above_reconstruction_tolerance = int(
    np.count_nonzero(
        absolute_errors > RECONSTRUCTION_TOLERANCE
    )
)


print("\nCompared measurements:", n_valid)


print(
    "Missingness disagreements:",
    missingness_disagreements
)


print(
    "Median absolute difference:",
    f"{median_absolute_error:.12g}"
)


print(
    "95th percentile absolute difference:",
    f"{p95_absolute_error:.12g}"
)


print(
    "Maximum absolute difference:",
    f"{max_absolute_error:.12g}"
)


print(
    "Measurements differing by more than 1e-6:",
    n_above_previous_tolerance
)


print(
    "Permitted numerical tolerance:",
    RECONSTRUCTION_TOLERANCE
)


print(
    "Measurements exceeding permitted tolerance:",
    n_above_reconstruction_tolerance
)


# ------------------------------------------------------------
# 6A. Validate missingness
# ------------------------------------------------------------

if missingness_disagreements != 0:

    raise RuntimeError(
        "Reported and reconstructed missingness differ."
    )


# ------------------------------------------------------------
# 6B. Validate numerical differences
# ------------------------------------------------------------

if max_absolute_error > RECONSTRUCTION_TOLERANCE:

    raise RuntimeError(
        "Reconstructed measurements exceed "
        "the permitted numerical tolerance."
    )


print("\nFull overlap correspondence: PASSED")


print(
    "Small numerical differences have been "
    "documented, not discarded."
)


# ------------------------------------------------------------
# 7. EXTENDED MSS PROTEIN STATISTICS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("EXTENDED MSS PROTEIN STATISTICS")
print("-" * 65)


original_gene_set = set(
    reported.index
)


records = []


for gene_symbol, row in reconstructed.iterrows():

    values = row.to_numpy(
        dtype=float
    )

    observed_values = values[
        np.isfinite(values)
    ]

    n = len(observed_values)


    median_ratio = np.nan
    fraction_positive = np.nan

    positive_count = 0
    negative_count = 0


    if n > 0:

        median_ratio = float(
            np.median(observed_values)
        )

        fraction_positive = float(
            np.mean(observed_values > 0)
        )

        positive_count = int(
            np.count_nonzero(
                observed_values > 0
            )
        )

        negative_count = int(
            np.count_nonzero(
                observed_values < 0
            )
        )


    pvalue = np.nan


    if n >= MIN_OBSERVATIONS:

        if np.all(observed_values == 0):

            pvalue = 1.0

        else:

            pvalue = float(
                wilcoxon(
                    observed_values,
                    alternative="two-sided",
                    zero_method="wilcox",
                    method="auto"
                ).pvalue
            )


    records.append({

        "protein_gene_symbol": gene_symbol,

        "in_original_paired_matrix":
            gene_symbol in original_gene_set,

        "mss_complete_pairs": n,

        "mss_missing_pairs":
            len(mss_patients) - n,

        "measurement_fraction":
            n / len(mss_patients),

        "median_tumor_normal_log2ratio":
            median_ratio,

        "fraction_tumor_higher":
            fraction_positive,

        "positive_patients":
            positive_count,

        "negative_patients":
            negative_count,

        "wilcoxon_pvalue":
            pvalue

    })


extended = pd.DataFrame(
    records
)


print("\nTotal protein rows:", len(extended))


print(
    "Proteins with >=30 complete MSS pairs:",
    int(
        extended["wilcoxon_pvalue"]
        .notna()
        .sum()
    )
)


newly_testable = extended[
    ~extended["in_original_paired_matrix"]
    & extended["wilcoxon_pvalue"].notna()
].copy()


print(
    "Additional proteins eligible for testing:",
    len(newly_testable)
)


# ------------------------------------------------------------
# 8. EXPANDED MULTIPLE-TESTING CORRECTION
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("MULTIPLE-TESTING CORRECTION")
print("-" * 65)


extended["wilcoxon_fdr"] = np.nan


testable = (
    extended["wilcoxon_pvalue"].notna()
)


if not testable.any():

    raise RuntimeError(
        "No eligible protein tests."
    )


adjusted = multipletests(

    extended.loc[
        testable,
        "wilcoxon_pvalue"
    ].to_numpy(dtype=float),

    method="fdr_bh"

)[1]


extended.loc[
    testable,
    "wilcoxon_fdr"
] = adjusted


# ------------------------------------------------------------
# 8A. Apply exploratory enrichment criteria
# ------------------------------------------------------------

extended["protein_enriched"] = (

    extended["mss_complete_pairs"].ge(
        MIN_OBSERVATIONS
    )

    & extended["wilcoxon_fdr"].lt(0.05)

    & extended[
        "median_tumor_normal_log2ratio"
    ].ge(1.0)

    & extended[
        "fraction_tumor_higher"
    ].ge(0.70)

)


print(
    "\nProteins meeting expanded enrichment criteria:",
    int(
        extended["protein_enriched"].sum()
    )
)


print(
    "Newly testable proteins meeting criteria:",
    int(
        (
            ~extended["in_original_paired_matrix"]
            & extended["protein_enriched"]
        ).sum()
    )
)


# ------------------------------------------------------------
# 9. COMPARE WITH ORIGINAL CPTAC RESULTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("ORIGINAL ANALYSIS CONSISTENCY")
print("-" * 65)


original = pd.read_csv(
    ORIGINAL_PROTEIN_PATH
)


if original[
    "protein_gene_symbol"
].duplicated().any():

    raise RuntimeError(
        "Duplicate original protein symbols."
    )


comparison = original[
    [
        "protein_gene_symbol",
        "mss_measurements",
        "median_tumor_normal_log2ratio",
        "protein_enriched"
    ]
].merge(

    extended[
        [
            "protein_gene_symbol",
            "mss_complete_pairs",
            "median_tumor_normal_log2ratio",
            "protein_enriched"
        ]
    ],

    on="protein_gene_symbol",

    how="left",

    suffixes=(
        "_original",
        "_extended"
    ),

    validate="one_to_one"

)


if len(comparison) != 6422:

    raise RuntimeError(
        "Original protein universe changed."
    )


# ------------------------------------------------------------
# 9A. Validate measurement coverage
# ------------------------------------------------------------

if not np.array_equal(

    comparison[
        "mss_measurements"
    ].to_numpy(),

    comparison[
        "mss_complete_pairs"
    ].to_numpy()

):

    raise RuntimeError(
        "Original and extended coverage differ."
    )


# ------------------------------------------------------------
# 9B. Validate median protein differences
#
# CORRECTED:
# Use the same reconstruction tolerance
# established in Section 6.
# ------------------------------------------------------------

original_medians = comparison[
    "median_tumor_normal_log2ratio_original"
].to_numpy(
    dtype=float
)


extended_medians = comparison[
    "median_tumor_normal_log2ratio_extended"
].to_numpy(
    dtype=float
)


if not np.allclose(

    original_medians,

    extended_medians,

    equal_nan=True,

    atol=RECONSTRUCTION_TOLERANCE,

    rtol=0.0

):

    raise RuntimeError(
        "Original and reconstructed protein "
        "medians differ beyond permitted tolerance."
    )


print("\nOriginal protein coverage: PASSED")

print("Original median ratios: PASSED")


original_enriched = (
    comparison[
        "protein_enriched_original"
    ].astype(bool)
)


extended_enriched = (
    comparison[
        "protein_enriched_extended"
    ].astype(bool)
)


print(
    "\nOriginal enriched proteins:",
    int(original_enriched.sum())
)


print(
    "Original enriched proteins retaining "
    "expanded-analysis criteria:",
    int(
        (
            original_enriched
            & extended_enriched
        ).sum()
    )
)


print(
    "\nNOTE: Expanded multiple-testing correction "
    "may change adjusted p-values."
)


# ------------------------------------------------------------
# 10. INTEGRATE TRANSCRIPTOMIC SCREENING GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("TRANSCRIPTOMIC / PROTEOMIC INTEGRATION")
print("-" * 65)


transcriptomic = pd.read_csv(
    TRANSCRIPTOMIC_PATH,
    dtype={"gene_id": str}
)


required_columns = {
    "gene_id",
    "gene_name",
    "combined_transcriptomic_support"
}


if not required_columns.issubset(
    transcriptomic.columns
):

    raise RuntimeError(
        "Unexpected transcriptomic table structure."
    )


supported_mask = (

    transcriptomic[
        "combined_transcriptomic_support"
    ]

    .astype(str)

    .str.lower()

    .eq("true")

)


supported = transcriptomic[
    supported_mask
].copy()


if len(supported) != 948:

    raise RuntimeError(
        "Expected 948 transcriptomic screening genes."
    )


supported["gene_name"] = (
    supported["gene_name"]
    .astype(str)
    .str.strip()
)


integrated = supported.merge(

    extended,

    left_on="gene_name",

    right_on="protein_gene_symbol",

    how="left",

    validate="many_to_one"

)


if len(integrated) != 948:

    raise RuntimeError(
        "Transcriptomic row count changed."
    )


covered = (
    integrated[
        "protein_gene_symbol"
    ].notna()
)


testable_integrated = (
    integrated[
        "wilcoxon_pvalue"
    ].notna()
)


enriched_integrated = (

    integrated[
        "protein_enriched"
    ]

    .fillna(False)

    .astype(bool)

)


print("\nTranscriptomic screening genes:", len(integrated))


print(
    "Represented in expanded assay:",
    int(covered.sum())
)


print(
    "With >=30 MSS paired observations:",
    int(testable_integrated.sum())
)


print(
    "Meeting expanded protein enrichment criteria:",
    int(enriched_integrated.sum())
)


additional_supported = integrated[

    enriched_integrated

    & ~integrated[
        "in_original_paired_matrix"
    ]
    .fillna(False)
    .astype(bool)

].copy()


print(
    "New screening genes supported "
    "outside original paired matrix:",
    len(additional_supported)
)


# ------------------------------------------------------------
# 11. REVIEW LEADING GENES
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("LEADING PROTEIN COVERAGE REVIEW")
print("-" * 65)


leading_genes = [
    "CDH3",
    "CLDN1",
    "CEMIP",
    "INHBA",
    "WNT2",
    "KRT80",
    "FAP",
    "DPEP1",
    "SLC7A5",
    "SLC5A6",
    "CTHRC1",
    "S100P"
]


leading = integrated[
    integrated["gene_name"].isin(
        leading_genes
    )
].copy()


order_lookup = {

    gene: index

    for index, gene in enumerate(
        leading_genes
    )

}


leading["display_order"] = (
    leading["gene_name"]
    .map(order_lookup)
)


leading = leading.sort_values(
    "display_order"
)


print("\n")


print(

    leading[
        [
            "gene_name",
            "in_original_paired_matrix",
            "mss_complete_pairs",
            "median_tumor_normal_log2ratio",
            "fraction_tumor_higher",
            "wilcoxon_fdr",
            "protein_enriched"
        ]
    ]

    .to_string(
        index=False,
        float_format=lambda x: f"{x:.4g}"
    )

)


print("\nIMPORTANT:")

print(
    "Proteins with fewer than 30 paired "
    "measurements remain insufficiently "
    "covered for formal enrichment testing."
)


# ------------------------------------------------------------
# 12. GENERATE PROTEIN COVERAGE FIGURE
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("GENERATING PROTEIN COVERAGE FIGURE")
print("-" * 65)


original_coverage = extended.loc[
    extended["in_original_paired_matrix"],
    "mss_complete_pairs"
].to_numpy()


additional_coverage = extended.loc[
    ~extended["in_original_paired_matrix"],
    "mss_complete_pairs"
].to_numpy()


fig, ax = plt.subplots(
    figsize=(10, 6)
)


bins = np.arange(
    -0.5,
    74.5,
    3
)


ax.hist(
    original_coverage,
    bins=bins,
    alpha=0.6,
    label="Original paired-matrix proteins"
)


ax.hist(
    additional_coverage,
    bins=bins,
    alpha=0.6,
    label="Additional protein rows"
)


ax.axvline(
    MIN_OBSERVATIONS,
    linestyle="--",
    linewidth=1,
    label="Minimum testing coverage"
)


ax.set_xlabel(
    "Complete MSS tumor-normal patient pairs"
)


ax.set_ylabel(
    "Number of proteins"
)


ax.set_title(
    "CPTAC Colon — Extended Protein Coverage"
)


ax.legend()


ax.grid(
    axis="y",
    alpha=0.2
)


fig.tight_layout()


figure_path = (
    FIGURES /
    "cptac_mss_extended_protein_coverage.png"
)


fig.savefig(
    figure_path,
    dpi=200,
    bbox_inches="tight"
)


plt.show()
plt.close(fig)


print("\nCoverage figure saved.")


# ------------------------------------------------------------
# 13. SAVE ANALYSIS OUTPUTS
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("SAVING EXTENDED PROTEOMICS RESULTS")
print("-" * 65)


extended.to_csv(

    RESULTS /
    "cptac_mss_extended_protein_statistics.csv",

    index=False

)


integrated.to_csv(

    RESULTS /
    "cptac_mss_extended_transcriptomic_integration.csv",

    index=False

)


newly_testable.to_csv(

    RESULTS /
    "cptac_mss_newly_testable_proteins.csv",

    index=False

)


additional_supported.to_csv(

    RESULTS /
    "cptac_mss_additional_supported_genes.csv",

    index=False

)


leading.to_csv(

    RESULTS /
    "cptac_mss_extended_leading_gene_review.csv",

    index=False

)


comparison.to_csv(

    RESULTS /
    "cptac_mss_original_extended_consistency.csv",

    index=False

)


print("\nAll extended analysis tables saved.")


# ------------------------------------------------------------
# 14. SAVE RESEARCH PROVENANCE
# ------------------------------------------------------------

provenance = {

    "project": "SurfaceTarget-CRC",

    "patch": "2.4",

    "patch_version": "2.4-corrected",

    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),

    "study": "CPTAC Prospective Colon Cancer",

    "source": "LinkedOmics",

    "analysis": (
        "Extended paired protein coverage "
        "and MSS enrichment using separate "
        "tumor and normal TMT log-ratio matrices"
    ),

    "published_mss_patients": int(
        len(mss_patients)
    ),

    "protein_rows_assessed": int(
        len(extended)
    ),

    "reported_overlap_measurements": int(
        n_valid
    ),

    "missingness_disagreements": int(
        missingness_disagreements
    ),

    # Numerical reconstruction audit

    "reconstruction_tolerance": float(
        RECONSTRUCTION_TOLERANCE
    ),

    "maximum_overlap_absolute_error": float(
        max_absolute_error
    ),

    "median_overlap_absolute_error": float(
        median_absolute_error
    ),

    "p95_overlap_absolute_error": float(
        p95_absolute_error
    ),

    "measurements_above_previous_1e-6_tolerance":
        int(n_above_previous_tolerance),

    "measurements_above_accepted_tolerance":
        int(n_above_reconstruction_tolerance),

    "reconstruction_validation_passed": True,

    # Statistical analysis

    "minimum_complete_pairs_for_testing":
        MIN_OBSERVATIONS,

    "proteins_tested": int(
        testable.sum()
    ),

    "newly_testable_proteins": int(
        len(newly_testable)
    ),

    "expanded_protein_enriched": int(
        extended["protein_enriched"].sum()
    ),

    # Cross-omics integration

    "transcriptomic_screening_genes": int(
        len(integrated)
    ),

    "expanded_protein_supported_screening_genes":
        int(enriched_integrated.sum()),

    "newly_supported_screening_genes":
        int(len(additional_supported)),

    # Statistical methods

    "statistical_test": (
        "Two-sided Wilcoxon signed-rank test"
    ),

    "multiple_testing": (
        "Benjamini-Hochberg across all "
        "eligible expanded protein tests"
    ),

    "protein_enrichment_criteria": {

        "minimum_mss_pairs": 30,

        "maximum_fdr": 0.05,

        "minimum_median_log2_difference": 1.0,

        "minimum_positive_fraction": 0.70
    },

    # Data handling

    "original_analysis_overwritten": False,

    "protein_imputation_performed": False,

    "tcga_cptac_patient_level_merging": False,

    # Scientific limitations

    "limitations": [

        "Extended values are reconstructed "
        "from the same CPTAC experiment.",

        "This is not independent replication.",

        "Small numerical differences between "
        "reported and reconstructed ratios "
        "are documented and bounded by tolerance.",

        "Sparse protein coverage limits inference.",

        "Missingness may be nonrandom.",

        "Only proteins with sufficient "
        "coverage receive statistical tests.",

        "Expanded testing changes the "
        "multiple-testing universe.",

        "Gene-symbol integration remains provisional.",

        "Tumor tissue includes malignant "
        "and nonmalignant cell populations.",

        "Protein enrichment does not prove "
        "cell-surface accessibility.",

        "No therapeutic targets have "
        "been validated."

    ]
}


with open(

    META /
    "cptac_mss_extended_analysis_provenance.json",

    "w"

) as f:

    json.dump(
        provenance,
        f,
        indent=4
    )


print("\nAnalysis provenance saved.")


# ------------------------------------------------------------
# 15. CREATE RESEARCH CHECKPOINT
# ------------------------------------------------------------

print("\n" + "-" * 65)
print("CREATING RESEARCH CHECKPOINT")
print("-" * 65)


timestamp = datetime.now(
    timezone.utc
).strftime("%Y%m%d_%H%M%S")


checkpoint = (

    CHECKPOINTS /

    f"SurfaceTarget_CRC_CPTAC_Patch24_{timestamp}.zip"

)


backup_folders = [
    "data/metadata",
    "data/processed",
    "data/raw/cptac_colon",
    "results/tables",
    "results/figures",
    "src",
    "reports",
    "logs"
]


with ZipFile(

    checkpoint,

    "w",

    ZIP_DEFLATED,

    compresslevel=3

) as archive:

    for folder in backup_folders:

        folder_path = ROOT / folder

        if not folder_path.exists():
            continue

        for file_path in folder_path.rglob("*"):

            if file_path.is_file():

                archive.write(

                    file_path,

                    arcname=str(
                        file_path.relative_to(ROOT)
                    )

                )


checkpoint_mib = (

    checkpoint.stat().st_size
    / (1024 ** 2)

)


print("\nCheckpoint created successfully.")


print(
    f"Checkpoint size: "
    f"{checkpoint_mib:.2f} MiB"
)


print("Checkpoint location:")
print(checkpoint)


# ------------------------------------------------------------
# 16. FINAL SUMMARY
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PATCH 2.4 SUMMARY")
print("=" * 65)


print("\nPublished MSS patients:", len(mss_patients))


print(
    "Total protein rows assessed:",
    len(extended)
)


print(
    "Proteins eligible for statistical testing:",
    int(testable.sum())
)


print(
    "Additional proteins eligible for testing:",
    len(newly_testable)
)


print(
    "Proteins meeting expanded enrichment criteria:",
    int(
        extended["protein_enriched"].sum()
    )
)


print(
    "\nTranscriptomic screening genes:",
    len(integrated)
)


print(
    "Screening genes with expanded protein support:",
    int(enriched_integrated.sum())
)


print(
    "New screening genes supported:",
    len(additional_supported)
)


print("\nCross-matrix validation: PASSED")


print(
    "Numerical tolerance:",
    RECONSTRUCTION_TOLERANCE
)


print(
    "Maximum reconstruction difference:",
    f"{max_absolute_error:.12g}"
)


print("\nSaved outputs:")


print("1. cptac_mss_extended_protein_statistics.csv")

print("2. cptac_mss_extended_transcriptomic_integration.csv")

print("3. cptac_mss_newly_testable_proteins.csv")

print("4. cptac_mss_additional_supported_genes.csv")

print("5. cptac_mss_extended_leading_gene_review.csv")

print("6. cptac_mss_original_extended_consistency.csv")

print("7. cptac_mss_extended_analysis_provenance.json")


print("\nFigure:")

print("cptac_mss_extended_protein_coverage.png")


print("\nResearch checkpoint: CREATED")


print(
    f"Checkpoint size: "
    f"{checkpoint_mib:.2f} MiB"
)


print("\nOriginal CPTAC results preserved.")

print("No source measurements modified.")

print("No protein imputation performed.")


print("\nPATCH 2.4 COMPLETE")

print("=" * 65)